# Debugging Spark — A Field Guide to the Spark UI

> **The Spark UI is your primary debugging tool for production performance issues.**

Every Spark job that runs — whether in Jupyter, via `spark-submit`, or on a managed platform like Databricks — exposes a web UI at `http://<driver-host>:4040`. The UI is live while the application runs and available as a history server after it finishes.

The single most common failure mode for Spark engineers is treating the UI as a black box. This notebook changes that. Each section pairs a real symptom you would see in production with the exact UI location to look at and the code-level fix to apply.

## How Spark organises work

Understanding the hierarchy prevents confusion when reading the UI:

```
Application
└── Job  (triggered by an action: .count(), .write(), .collect())
    └── Stage  (separated by shuffles)
        └── Task  (one task per partition — the atomic unit of work)
```

A slow *task* means data skew or executor-level resource contention. A slow *stage* usually means an expensive shuffle or missing cache. A slow *job* with fast stages usually means you have too many jobs (e.g., iterating in Python instead of using Spark).

## Symptom → UI Location → What to Check

| # | Symptom | UI Tab | What to look at | Likely cause |
|---|---------|--------|-----------------|-------------|
| 1 | **Job is slow overall** | **Jobs** | Stage timeline — which stage is the widest bar? | One stage dominates; often a shuffle-heavy aggregation |
| 2 | **One task is much slower than the rest** | **Stage detail** → Task duration histogram | Long right tail in the histogram; max task time >> median task time | Data skew — one partition has far more data |
| 3 | **OOM / executor dying** | **Executors** tab | GC Time column > 10% of task time; Peak Memory near executor limit | Heap pressure from large state, cartesian join, or too-small executor |
| 4 | **First run slow, second run fast** | **Storage** tab | Nothing cached on first run; dataset appears on second run | Cached RDD/DataFrame pays off only from run 2 onward |
| 5 | **Disk I/O unexpectedly high** | **Stage detail** | Shuffle Spill (Memory) and Shuffle Spill (Disk) columns | Insufficient shuffle memory; data being spilled to local disk |
| 6 | **Skewed join causing straggler tasks** | **Stage detail** | Task duration variance; median << max | Join key has a hot partition (e.g. NULL or a single dominant value) |

### How to navigate to Stage detail

1. Open `http://<driver>:4040`
2. Click **Jobs** → click the job ID
3. Click the stage number in the DAG or stage list
4. Scroll down to the **Tasks** table — sort by **Duration** descending to find the straggler
5. Click on the straggler task's executor link to see GC breakdown

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
import random

spark = (
    SparkSession.builder
    .appName("SparkUIDebuggingFieldGuide")
    .master("spark://spark-master:7077")
    # Keep shuffle partitions low — we want the skew to be dramatic and
    # visible in the UI without waiting for 200 mostly-empty partitions.
    .config("spark.sql.shuffle.partitions", "8")
    # Disable AQE (Adaptive Query Execution) so that skew manifests visibly.
    # In production you WANT AQE enabled — it auto-splits skewed partitions.
    # We disable it here so the debugging exercises are instructive.
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.sql.adaptive.skewJoin.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — UI at http://localhost:4040")
print("Note: AQE is intentionally disabled for this debugging demo.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SYMPTOM 2 & 6: Data skew causing a long-tail task
#
# We deliberately create a skewed dataset:
#   - 99 % of rows have joinKey = 1  (the "hot" key)
#   -  1 % of rows are spread across keys 2–100
#
# After a SortMergeJoin, all rows with joinKey=1 land in ONE partition.
# That task processes 99× more data than the others → it's the straggler.
#
# What you would see in the UI (Stage detail → task duration histogram):
#   Histogram: 7 bars near 0.1s, 1 tall bar at 5–10s  (the skewed task)
# ─────────────────────────────────────────────────────────────────────────────

TOTAL_ROWS = 200_000
HOT_ROWS   = int(TOTAL_ROWS * 0.99)  # 99% go to key=1
COLD_ROWS  = TOTAL_ROWS - HOT_ROWS

# Build the left table (transactions)
hot_data  = [(1, float(i), f"txn_{i}") for i in range(HOT_ROWS)]
cold_data = [(random.randint(2, 100), float(i), f"txn_{i}") for i in range(COLD_ROWS)]
left_rows = hot_data + cold_data

left_df = spark.createDataFrame(left_rows, ["joinKey", "amount", "txnId"])

# Build the right (dimension) table — small, one row per key
right_rows = [(k, f"category_{k}") for k in range(1, 101)]
right_df = spark.createDataFrame(right_rows, ["joinKey", "category"])

print(f"Left table: {left_df.count():,} rows")
print("Key distribution (top 5):")
left_df.groupBy("joinKey").count().orderBy(F.col("count").desc()).show(5)

# ── Perform the skewed SortMergeJoin ─────────────────────────────────────────
# Hint: force SortMergeJoin so Spark doesn't auto-broadcast the right side
from pyspark.sql.functions import broadcast

# No broadcast hint → Spark will do SortMergeJoin → skew is visible
skewed_result = left_df.join(right_df, on="joinKey", how="inner")

print("\nRunning skewed SortMergeJoin — watch the Stage detail in the UI ...")
skewed_result.groupBy("category").agg(
    F.count("*").alias("txnCount"),
    F.sum("amount").alias("totalAmount")
).orderBy(F.col("txnCount").desc()).show(10)

print("\nKey statistics to correlate with UI:")
skewed_result.select("amount").describe().show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FIX: SALTING — distributing the hot key across multiple partitions
#
# Idea:
#   1. Choose a salt factor N (how many partitions to spread the hot key across)
#   2. Add a random salt 0..N-1 to every row's join key:  new_key = (joinKey, salt)
#   3. Explode the RIGHT table: for each original right row, produce N copies
#      with salt 0, 1, …, N-1 so that every salted left row has a matching right row
#   4. Join on (joinKey, salt)
#   5. After the join, drop the salt — it was only needed for partitioning
#
# Cost: the right table grows N× in memory. Acceptable when the right table
# is a small dimension table (which is the common case for skewed joins).
# ─────────────────────────────────────────────────────────────────────────────

SALT_FACTOR = 8  # spread the hot key across 8 partitions

# Step 1 & 2: salt the left table
left_salted = left_df.withColumn(
    "salt",
    # Assign a random salt bucket 0 .. SALT_FACTOR-1 to each row
    (F.rand() * SALT_FACTOR).cast("int")
)

# Step 3: explode the right table — one copy per salt value
# F.array(lit(0), lit(1), ..., lit(N-1)) + F.explode → N rows per original row
salt_values = F.array(*[F.lit(i) for i in range(SALT_FACTOR)])
right_exploded = right_df.withColumn("salt", F.explode(salt_values))

print(f"Right table after explosion: {right_exploded.count()} rows (was {right_df.count()})")

# Step 4: join on composite key (joinKey, salt)
fixed_result = left_salted.join(
    right_exploded,
    on=["joinKey", "salt"],
    how="inner"
)

# Step 5: aggregate (same business logic as before) — salt is transparent to results
print("Running salted join — task duration histogram should be flat now ...")
fixed_result.groupBy("category").agg(
    F.count("*").alias("txnCount"),
    F.sum("amount").alias("totalAmount")
).orderBy(F.col("txnCount").desc()).show(10)

print("""
What to verify in the UI after running this cell:
  - Stage detail → task duration histogram should show a narrow, symmetric
    distribution instead of a long right tail.
  - Max task time / Median task time ratio should drop from ~50× to ~1.5×.
  - Overall stage wall-clock time should drop significantly.
""")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SYMPTOM 5: Shuffle Spill — what it is and how to trigger it
#
# When Spark sorts or hashes data for a shuffle, it uses the execution memory
# region of the executor JVM. If that region fills up before the operation
# completes, Spark spills the in-memory buffer to the executor's LOCAL DISK
# (not HDFS) as a temporary file.
#
# This is MUCH slower than in-memory operation (10-100×) and is a clear signal
# that either:
#   (a) Executors are under-provisioned (increase spark.executor.memory)
#   (b) Partition count is too low (increase spark.sql.shuffle.partitions)
#   (c) A data skew feeds too much data into one partition
#
# We simulate spill by shrinking the memory fraction to almost nothing.
# DO NOT do this in production — it's purely for educational demonstration.
# ─────────────────────────────────────────────────────────────────────────────

# Stop the current session and restart with a very small memory fraction
spark.stop()

spark_tight = (
    SparkSession.builder
    .appName("SpillDemo")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions", "4")
    # spark.memory.fraction controls how much of the executor heap Spark
    # can use for both execution (sort/hash) and storage (cache).
    # Default is 0.6. Setting it to 0.05 forces spill almost immediately.
    .config("spark.memory.fraction", "0.05")
    # Unified memory manager — execution and storage share this pool.
    .config("spark.memory.storageFraction", "0.1")
    .config("spark.sql.adaptive.enabled", "false")
    .getOrCreate()
)
spark_tight.sparkContext.setLogLevel("WARN")

# A large sort is the classic spill trigger — Spark must hold many records in
# memory while building the sort buffer.
spill_data = spark_tight.range(0, 500_000).withColumn(
    "payload",
    F.concat(F.lit("data_"), (F.rand() * 1_000_000).cast("long").cast("string"))
)

# This global sort forces all partitions to be read and merged in order.
print("Running memory-constrained sort — spill expected ...")
print("After this runs, check Stage detail → look for non-zero 'Shuffle Spill (Disk)' column")
sorted_count = spill_data.orderBy("payload").count()
print(f"Sorted row count: {sorted_count:,}")

# Show what the explain() looks like — note the Sort operator
print("\nPhysical plan for the sort:")
spill_data.orderBy("payload").explain()
# Look for: Sort [payload ASC NULLS FIRST], true, 0
# The 'true' means global sort (not local sort within partition).
# Global sorts always shuffle → spill risk is high on tight memory.

# Restore a normal session for the rest of the notebook
spark_tight.stop()

spark = (
    SparkSession.builder
    .appName("SparkUIDebuggingFieldGuide")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("\nRestored normal SparkSession.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# READING THE SQL TAB — parsing explain("formatted") output
#
# The Spark UI's SQL tab renders a visual DAG of the physical plan for every
# query. The same information is available programmatically via
# df.explain("formatted"), which is more scripting-friendly.
#
# This cell builds a multi-step query that exercises several interesting
# operators, then walks through the formatted explain output and highlights
# where to look for bottlenecks.
# ─────────────────────────────────────────────────────────────────────────────

# Recreate sample data
orders_data = [
    (i, (i % 50) + 1, float(i * 10 % 500), f"2024-01-{(i % 28) + 1:02d}")
    for i in range(1, 5001)
]
customers_data = [(k, f"customer_{k}", ["US", "UK", "DE", "FR"][k % 4]) for k in range(1, 51)]

orders_df    = spark.createDataFrame(orders_data,    ["orderId", "customerId", "amount", "orderDate"])
customers_df = spark.createDataFrame(customers_data, ["customerId", "name", "country"])

# ── Complex query: join + filter + window function + aggregation ──────────────
complex_query = (
    orders_df
    # Filter before join — good practice even though Catalyst would push it down
    .filter(F.col("amount") > 50)
    .join(customers_df, on="customerId", how="inner")
    # Window function: rank orders within each country by amount
    .withColumn(
        "rankInCountry",
        F.rank().over(
            # Window spec: partition by country, order by amount descending
            # This forces a shuffle (repartition by country) + sort within partition
            __import__("pyspark.sql.window", fromlist=["Window"]).Window
            .partitionBy("country")
            .orderBy(F.col("amount").desc())
        )
    )
    # Keep only top-5 orders per country
    .filter(F.col("rankInCountry") <= 5)
    .groupBy("country")
    .agg(
        F.count("orderId").alias("topOrderCount"),
        F.sum("amount").alias("topRevenue")
    )
    .orderBy("country")
)

print("=" * 80)
print("FORMATTED EXPLAIN — how to read it")
print("=" * 80)
complex_query.explain("formatted")

print("""
HOW TO READ THE FORMATTED EXPLAIN OUTPUT — field guide
═══════════════════════════════════════════════════════════════════════════════

The plan is read TOP-DOWN (execution flows bottom-up, but the indentation
makes top = final result, bottom = data source).

KEY OPERATOR PATTERNS TO WATCH FOR:

  Exchange (hashpartitioning / rangepartitioning)
  ▶ This is a SHUFFLE. Every Exchange is a stage boundary.
    Count the Exchanges — that's how many shuffles your query does.
    Rule of thumb: > 3 shuffles in a pipeline is a redesign opportunity.

  Sort [col ASC/DESC], true
  ▶ Global sort after a shuffle. The 'true' = global (not local).
    Global sorts are expensive — avoid on large datasets unless required.

  Window [partition by X, order by Y]
  ▶ Always triggers Exchange (shuffle) + Sort. If you see this for a
    high-cardinality partition key, watch for spill.

  BroadcastHashJoin
  ▶ GOOD — no shuffle. Spark broadcasts the small table.
    Threshold: spark.sql.autoBroadcastJoinThreshold (default 10 MB).

  SortMergeJoin
  ▶ Both sides are shuffled and sorted. Always 2 Exchanges.
    If one side is small, consider broadcast hint or increase the threshold.

  Filter [condition], pushed: [condition]
  ▶ 'pushed' means it was pushed to the file scan (Parquet row-group
    skipping). If your filter does NOT appear in 'pushed', Catalyst
    couldn't push it — check for type mismatches or UDF wrapping.

  *(N) prefix → WholeStageCodegen stage N
  ▶ Operators inside the same codegen stage are fused by Tungsten into
    a single JVM method. Faster than interpreted execution.

WORKFLOW FOR BOTTLENECK IDENTIFICATION:
  1. Count Exchange nodes → each one is a shuffle stage.
  2. Find the stage with the widest bar in the Jobs timeline (slowest stage).
  3. Map the stage number in the UI back to the Exchange node in explain() output.
  4. Look at what operator is ABOVE the Exchange — that's the one doing the work.
  5. Apply the relevant fix: broadcast hint, salting, partition adjustment, cache.
═══════════════════════════════════════════════════════════════════════════════
""")

# Run the query so we can inspect it in the SQL tab
result = complex_query.collect()
print(f"Query complete: {len(result)} rows in result")
for row in result:
    print(f"  {row['country']}: {row['topOrderCount']} orders, revenue ${row['topRevenue']:.2f}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CLEANUP
# ─────────────────────────────────────────────────────────────────────────────
# After stopping the session, the UI at :4040 becomes unavailable.
# The History Server (if configured) will retain all event logs at :18080.
# Point spark.eventLog.dir to an HDFS path and
# start the history server with: $SPARK_HOME/sbin/start-history-server.sh

spark.stop()
print("SparkSession stopped.")
print("Review your findings in the History Server if event logging was enabled.")